In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

CUDA available: True
CUDA device: NVIDIA A40
Using device: cuda


In [3]:
# Define paths
original_repo = '/net/scratch2/smallyan/modular_addition_claude_2025-12-26_01-20-45'
replication_dir = os.path.join(original_repo, 'evaluation/replications')

# Check if directories exist
print(f"Original repo exists: {os.path.exists(original_repo)}")
print(f"Replication dir exists: {os.path.exists(replication_dir)}")

# List contents of original repo
print("\nOriginal repo contents:")
if os.path.exists(original_repo):
    for item in os.listdir(original_repo):
        print(f"  {item}")

Original repo exists: True
Replication dir exists: True

Original repo contents:
  no_exe_evaluation
  results
  real_circuits_1.json
  notebooks
  logs
  evaluation
  .git
  doc_only_evaluation


In [4]:
# Look for original documentation
print("Looking for original documentation.md...")
for root, dirs, files in os.walk(original_repo):
    for f in files:
        if 'documentation' in f.lower() and f.endswith('.md'):
            full_path = os.path.join(root, f)
            print(f"Found: {full_path}")

Looking for original documentation.md...
Found: /net/scratch2/smallyan/modular_addition_claude_2025-12-26_01-20-45/logs/documentation.md
Found: /net/scratch2/smallyan/modular_addition_claude_2025-12-26_01-20-45/evaluation/replications/documentation_replication.md
Found: /net/scratch2/smallyan/modular_addition_claude_2025-12-26_01-20-45/evaluation/replication_eval/documentation_evaluation_summary.md


In [5]:
# Read original documentation
original_doc_path = '/net/scratch2/smallyan/modular_addition_claude_2025-12-26_01-20-45/logs/documentation.md'
with open(original_doc_path, 'r') as f:
    original_doc = f.read()
print("=== ORIGINAL DOCUMENTATION ===")
print(original_doc)

=== ORIGINAL DOCUMENTATION ===
# Modular Addition Circuit Documentation

## 1. Goal

Reverse-engineer the modular addition circuit in a 1-layer transformer trained to compute `(a + b) mod p` where p=113 (prime). The objective is to understand how the model implements this arithmetic operation and verify whether it uses a Discrete Fourier Transform (DFT) based algorithm.

## 2. Hypothesis

The modular addition circuit implements a **Discrete Fourier Transform algorithm**:

1. **Embedding Layer**: Encodes input numbers `a` and `b` as Fourier features
   - Each number represented as `cos(2πka/p)` and `sin(2πka/p)` for multiple frequencies `k`

2. **Attention Heads**: Combine Fourier representations of `a` and `b`
   - Move Fourier features from input positions to the output position
   - Enable the MLP to access both inputs' representations

3. **MLP Layer**: Implements angle addition using trigonometric identities
   - `cos(k(a+b)) = cos(ka)cos(kb) - sin(ka)sin(kb)`
   - `sin(k(a+b)) = s

In [6]:
# Read replicated documentation
replication_doc_path = '/net/scratch2/smallyan/modular_addition_claude_2025-12-26_01-20-45/evaluation/replications/documentation_replication.md'
with open(replication_doc_path, 'r') as f:
    replication_doc = f.read()
print("=== REPLICATED DOCUMENTATION ===")
print(replication_doc)

=== REPLICATED DOCUMENTATION ===
# Modular Addition Circuit Replication - Documentation

## 1. Goal

Replicate the modular addition circuit analysis experiment to reverse-engineer how a 1-layer transformer computes `(a + b) mod p` where p=113 (prime). The objective is to verify whether the model uses a Discrete Fourier Transform (DFT) based algorithm.

## 2. Data

- **Task**: Modular addition `(a + b) mod p`
- **Modulus**: p = 113 (prime)
- **Dataset size**: 12,769 examples (113² = all pairs)
- **Train/Test split**: 80/20 (10,215 train, 2,554 test)
- **Input format**: [a, b, =] where a, b ∈ {0, 1, ..., 112}
- **Output**: c = (a + b) mod 113
- **Random seed**: 42 (for reproducibility)

Example:
```
Input:  [42, 67, =]
Output: 109  (since (42 + 67) mod 113 = 109)
```

## 3. Method

### Phase 1: Model Training
- Trained a 1-layer transformer with 4 attention heads
- Model architecture: d_model=128, d_mlp=512, n_heads=4
- Training:
  - Phase 1: 200 epochs with AdamW (lr=1e-3, weight_decay=

In [7]:
# Now let's perform the evaluation

import json
from datetime import datetime

# Create output directory
output_dir = os.path.join(original_repo, 'evaluation/new_replication_eval')
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory created: {output_dir}")

Output directory created: /net/scratch2/smallyan/modular_addition_claude_2025-12-26_01-20-45/evaluation/new_replication_eval


In [8]:
# Evaluate DE1: Result Fidelity

# Key metrics to compare:
print("=" * 60)
print("DE1: RESULT FIDELITY EVALUATION")
print("=" * 60)

# 1. Test Accuracy
orig_accuracy = 99.45
repl_accuracy = 99.92
accuracy_diff = abs(repl_accuracy - orig_accuracy)
accuracy_pct_diff = (accuracy_diff / orig_accuracy) * 100
print(f"\n1. Test Accuracy:")
print(f"   Original: {orig_accuracy}%")
print(f"   Replicated: {repl_accuracy}%")
print(f"   Difference: {accuracy_diff:.2f}% (absolute)")
print(f"   Percentage deviation: {accuracy_pct_diff:.2f}%")

# 2. Grokking phenomenon
print(f"\n2. Grokking Phenomenon:")
print(f"   Original: Yes")
print(f"   Replicated: Yes")
print(f"   Match: YES")

# 3. MLP Fourier Correlation (key quantitative result)
print(f"\n3. MLP Fourier Correlation:")
print(f"   Original: 0.93-0.96")
print(f"   Replicated: 0.87-0.98")
print(f"   Overlap in range: YES (both show very high correlation)")

# 4. Embedding Fourier correlations
print(f"\n4. Embedding Fourier Correlations:")
print(f"   Original: 0.75-0.87")
print(f"   Replicated: 0.37-0.78")
print(f"   Analysis: Replication shows slightly lower but still meaningful correlations")

# 5. Ablation results - both show all components essential
print(f"\n5. Ablation Results:")
print("   Original - All components cause near-total accuracy drop when removed:")
print("     No Attention: 0.98%, No MLP: 2.11%")
print("   Replicated - Same pattern:")
print("     No Attention: 0.78%, No MLP: 1.17%")
print("   Match: YES (same qualitative findings)")

# 6. Circuit structure
print(f"\n6. Circuit Structure:")
print('   Original: ["input", "a0.h0", "a0.h1", "a0.h2", "a0.h3", "m0"]')
print('   Replicated: ["input", "a0.h0", "a0.h1", "a0.h2", "a0.h3", "m0"]')
print("   Match: EXACT")

# Calculate overall pass/fail
de1_pass = True
de1_reasons = []

# Check test accuracy (within 5%)
if accuracy_pct_diff <= 5:
    de1_reasons.append(f"Test accuracy within tolerance ({accuracy_pct_diff:.2f}% < 5%)")
else:
    de1_pass = False
    de1_reasons.append(f"Test accuracy outside tolerance ({accuracy_pct_diff:.2f}% > 5%)")

# Check qualitative findings match
de1_reasons.append("Grokking phenomenon: Both report YES")
de1_reasons.append("All components essential: Both confirm")
de1_reasons.append("MLP Fourier correlations: Both show very high (>0.87)")
de1_reasons.append("Circuit structure: Exact match")

print(f"\n{'=' * 60}")
print(f"DE1 RESULT: {'PASS' if de1_pass else 'FAIL'}")
print("=" * 60)

DE1: RESULT FIDELITY EVALUATION

1. Test Accuracy:
   Original: 99.45%
   Replicated: 99.92%
   Difference: 0.47% (absolute)
   Percentage deviation: 0.47%

2. Grokking Phenomenon:
   Original: Yes
   Replicated: Yes
   Match: YES

3. MLP Fourier Correlation:
   Original: 0.93-0.96
   Replicated: 0.87-0.98
   Overlap in range: YES (both show very high correlation)

4. Embedding Fourier Correlations:
   Original: 0.75-0.87
   Replicated: 0.37-0.78
   Analysis: Replication shows slightly lower but still meaningful correlations

5. Ablation Results:
   Original - All components cause near-total accuracy drop when removed:
     No Attention: 0.98%, No MLP: 2.11%
   Replicated - Same pattern:
     No Attention: 0.78%, No MLP: 1.17%
   Match: YES (same qualitative findings)

6. Circuit Structure:
   Original: ["input", "a0.h0", "a0.h1", "a0.h2", "a0.h3", "m0"]
   Replicated: ["input", "a0.h0", "a0.h1", "a0.h2", "a0.h3", "m0"]
   Match: EXACT

DE1 RESULT: PASS


In [9]:
# Evaluate DE2: Conclusion Consistency

print("=" * 60)
print("DE2: CONCLUSION CONSISTENCY EVALUATION")
print("=" * 60)

print("\n1. Main Hypothesis:")
print("   Original: DFT-based algorithm is SUPPORTED")
print("   Replicated: DFT-based algorithm is SUPPORTED")
print("   Match: YES")

print("\n2. Evidence Supporting Hypothesis:")
print("   Original claims:")
print("     - Fourier features in embeddings")
print("     - Angle addition in MLP")
print("     - All components necessary")
print("     - Key frequencies related to prime modulus")
print("   Replicated claims:")
print("     - Fourier features in embeddings")
print("     - Angle addition in MLP")
print("     - All components necessary")
print("     - Key frequencies related to prime modulus")
print("   Match: YES (all four evidence types confirmed)")

print("\n3. Circuit Mechanism Description:")
print("   Both documents describe the same 4-step mechanism:")
print("     1. Embedding encodes numbers as Fourier features")
print("     2. Attention moves Fourier representations to = position")
print("     3. MLP computes angle addition using trig identities")
print("     4. Unembedding decodes to output token")
print("   Match: YES")

print("\n4. Interpretation of Results:")
print("   Original: 'All components are essential'")
print("   Replicated: 'All components are critical'")
print("   Both conclude the model uses full capacity")
print("   Match: YES")

print("\n5. Note on Frequency Differences:")
print("   Original key frequencies: k = 15, 49, 51, 52")
print("   Replicated key frequencies: k = 26, 36, 52")
print("   Both documents explain this is expected due to random initialization")
print("   The replication explicitly states this in 'Comparison with Original Results'")
print("   Match: YES (difference explained, core mechanism confirmed)")

de2_pass = True
de2_rationale = (
    "Both documents conclude the DFT-based algorithm hypothesis is SUPPORTED. "
    "The four key evidence points are identical: Fourier features in embeddings, "
    "angle addition in MLP, all components necessary, and key frequencies related to prime modulus. "
    "The circuit mechanism is described identically. The replication correctly notes that "
    "specific frequencies differ due to random initialization but the core mechanism is confirmed."
)

print(f"\n{'=' * 60}")
print(f"DE2 RESULT: {'PASS' if de2_pass else 'FAIL'}")
print("=" * 60)

DE2: CONCLUSION CONSISTENCY EVALUATION

1. Main Hypothesis:
   Original: DFT-based algorithm is SUPPORTED
   Replicated: DFT-based algorithm is SUPPORTED
   Match: YES

2. Evidence Supporting Hypothesis:
   Original claims:
     - Fourier features in embeddings
     - Angle addition in MLP
     - All components necessary
     - Key frequencies related to prime modulus
   Replicated claims:
     - Fourier features in embeddings
     - Angle addition in MLP
     - All components necessary
     - Key frequencies related to prime modulus
   Match: YES (all four evidence types confirmed)

3. Circuit Mechanism Description:
   Both documents describe the same 4-step mechanism:
     1. Embedding encodes numbers as Fourier features
     2. Attention moves Fourier representations to = position
     3. MLP computes angle addition using trig identities
     4. Unembedding decodes to output token
   Match: YES

4. Interpretation of Results:
   Original: 'All components are essential'
   Replicated: